In [0]:
from pyspark.sql.functions import hour, dayofweek, month, col, lag
from pyspark.sql.window import Window

# Load Silver tables
silver_power = spark.read.table("default.silver_power")
silver_weather = spark.read.table("default.silver_weather")

# 1. Define the Window (Order by time)
window_spec = Window.orderBy("timestamp")

# 2. Join and Engineer Features
gold_df = (
    silver_power.join(silver_weather, on="timestamp", how="inner")
    .withColumn("hour", hour(col("timestamp")))
    .withColumn("day_of_week", dayofweek(col("timestamp")))
    .withColumn("month", month(col("timestamp")))
    # THE KEY FEATURE: Look back 24 hours for the previous demand reading
    .withColumn("demand_lag_24h", lag(col("demand_mw"), 24).over(window_spec))
    # Drop rows where lag is null (the first 24 hours of your dataset)
    .dropna(subset=["demand_lag_24h"])
    .select(
        "timestamp", 
        "hour", 
        "day_of_week", 
        "month", 
        "temperature", 
        "precipitation", 
        "demand_lag_24h", # Our new feature
        "demand_mw"       # Our target
    )
)

# Overwrite the Gold Table with the new schema
(gold_df.write
  .format("delta")
  .mode("overwrite")
  .option("overwriteSchema", "true") # Use this instead of mergeSchema for overwrites
  .saveAsTable("default.gold_power_demand"))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
